# ✂️ Advanced Chunking Strategies

**Optimal text splitting for better RAG retrieval**

---

## 📋 Overview

**What you'll learn:**
- Chunking strategies comparison
- Semantic chunking
- Context-aware splitting
- Chunk overlap strategies
- Optimal chunk size selection

**Time estimate:** ⏱️ 50 minutes | **Difficulty:** 🔴 Advanced

---

In [ ]:
from langchain.text_splitter import RecursiveCharacterTextSplitter, CharacterTextSplitter
from sentence_transformers import SentenceTransformer
import numpy as np
from typing import List, Dict
import matplotlib.pyplot as plt

print("✅ Setup complete")

## 🤔 Why Chunking Matters

### Problem:
```
Document: 50,000 words
LLM context: 4,000 tokens
Can't fit entire document!
```

### Solution: Chunk it
```
Document → [Chunk 1] [Chunk 2] ... [Chunk 100]
           ↓
Retrieve relevant chunks only
```

### The Goldilocks Problem:
- 🐻 **Too small**: Loses context
- 🐻 **Too large**: Irrelevant info, expensive
- 🎯 **Just right**: Complete ideas, focused

**Typical range**: 200-500 tokens per chunk

## 📊 Chunking Strategies Comparison

In [ ]:
# Sample document
sample_doc = """Python is a high-level programming language known for its simplicity.

Python supports multiple programming paradigms including procedural, object-oriented, and functional programming.

Popular frameworks like Django and Flask make web development easy. NumPy and Pandas are essential for data science.

Machine learning libraries like TensorFlow and PyTorch have made Python the go-to language for AI development."""

print("📄 Sample Document:")
print(sample_doc)
print(f"\nLength: {len(sample_doc)} characters")

In [ ]:
# Strategy 1: Fixed-size chunking
def fixed_size_chunking(text: str, chunk_size: int = 100, overlap: int = 20) -> List[str]:
    """Simple fixed-size chunks."""
    chunks = []
    for i in range(0, len(text), chunk_size - overlap):
        chunk = text[i:i + chunk_size]
        if chunk:
            chunks.append(chunk)
    return chunks

# Strategy 2: Sentence-based
def sentence_chunking(text: str, sentences_per_chunk: int = 3) -> List[str]:
    """Chunk by sentences."""
    sentences = text.split('. ')
    chunks = []
    
    for i in range(0, len(sentences), sentences_per_chunk):
        chunk = '. '.join(sentences[i:i + sentences_per_chunk])
        chunks.append(chunk)
    
    return chunks

# Strategy 3: Paragraph-based
def paragraph_chunking(text: str) -> List[str]:
    """Chunk by paragraphs."""
    return [p.strip() for p in text.split('\n\n') if p.strip()]

# Strategy 4: Recursive (best!)
def recursive_chunking(text: str, chunk_size: int = 200, overlap: int = 50) -> List[str]:
    """Recursive character splitting."""
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=overlap,
        separators=["\n\n", "\n", ". ", " ", ""]
    )
    return splitter.split_text(text)

# Compare
print("\n📊 Chunking Strategies Comparison\n")
print("="*70)

strategies = [
    ("Fixed-size", fixed_size_chunking(sample_doc, 100, 20)),
    ("Sentence", sentence_chunking(sample_doc, 2)),
    ("Paragraph", paragraph_chunking(sample_doc)),
    ("Recursive", recursive_chunking(sample_doc, 150, 30)),
]

for name, chunks in strategies:
    print(f"\n{name}: {len(chunks)} chunks")
    print(f"First chunk ({len(chunks[0])} chars):")
    print(f"  '{chunks[0][:80]}...'") if len(chunks[0]) > 80 else print(f"  '{chunks[0]}'")

## 🎯 Semantic Chunking

In [ ]:
class SemanticChunker:
    """Chunk based on semantic similarity."""
    
    def __init__(self, model_name: str = 'all-MiniLM-L6-v2'):
        self.model = SentenceTransformer(model_name)
    
    def chunk(self, text: str, similarity_threshold: float = 0.5) -> List[str]:
        """Split at points where semantic similarity drops."""
        # Split into sentences first
        sentences = [s.strip() + '.' for s in text.split('. ') if s.strip()]
        
        if len(sentences) <= 1:
            return sentences
        
        # Get embeddings
        embeddings = self.model.encode(sentences)
        
        # Calculate similarities between consecutive sentences
        similarities = []
        for i in range(len(embeddings) - 1):
            sim = np.dot(embeddings[i], embeddings[i+1]) / (
                np.linalg.norm(embeddings[i]) * np.linalg.norm(embeddings[i+1])
            )
            similarities.append(sim)
        
        # Find split points (where similarity drops)
        split_points = [0]
        for i, sim in enumerate(similarities):
            if sim < similarity_threshold:
                split_points.append(i + 1)
        split_points.append(len(sentences))
        
        # Create chunks
        chunks = []
        for i in range(len(split_points) - 1):
            chunk = ' '.join(sentences[split_points[i]:split_points[i+1]])
            chunks.append(chunk)
        
        return chunks

# Test semantic chunking
semantic_chunker = SemanticChunker()
semantic_chunks = semantic_chunker.chunk(sample_doc, similarity_threshold=0.6)

print("🎯 Semantic Chunking Results\n")
print(f"Number of chunks: {len(semantic_chunks)}\n")
for i, chunk in enumerate(semantic_chunks, 1):
    print(f"Chunk {i}:")
    print(f"  {chunk}\n")

## 🔄 Chunk Overlap Strategies

In [ ]:
def visualize_overlap(text: str, chunk_size: int, overlap: int):
    """Visualize chunk overlap."""
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=overlap
    )
    chunks = splitter.split_text(text)
    
    print(f"Chunk size: {chunk_size}, Overlap: {overlap}\n")
    print(f"Total chunks: {len(chunks)}\n")
    
    for i, chunk in enumerate(chunks[:3], 1):  # Show first 3
        print(f"Chunk {i} ({len(chunk)} chars):")
        print(f"  '{chunk[:60]}...'")
        
        # Show overlap with next chunk
        if i < len(chunks):
            next_chunk = chunks[i]
            # Find overlap (simple approximation)
            overlap_text = ""
            for j in range(min(overlap, len(chunk))):
                if chunk[-overlap:] == next_chunk[:overlap]:
                    overlap_text = chunk[-overlap:]
                    break
            
            if overlap_text:
                print(f"  Overlap with next: '{overlap_text[:40]}...'")
        print()

# Test different overlap amounts
print("🔄 Chunk Overlap Visualization\n")
print("="*70)

print("\n1. No Overlap (overlap=0):")
visualize_overlap(sample_doc, chunk_size=150, overlap=0)

print("\n2. Small Overlap (overlap=30):")
visualize_overlap(sample_doc, chunk_size=150, overlap=30)

print("\n💡 Overlap preserves context across chunks!")

## 📏 Finding Optimal Chunk Size

In [ ]:
def evaluate_chunk_size(text: str, chunk_sizes: List[int]) -> Dict:
    """Evaluate different chunk sizes."""
    results = []
    
    for size in chunk_sizes:
        splitter = RecursiveCharacterTextSplitter(
            chunk_size=size,
            chunk_overlap=int(size * 0.2)  # 20% overlap
        )
        chunks = splitter.split_text(text)
        
        avg_length = np.mean([len(c) for c in chunks])
        
        results.append({
            'chunk_size': size,
            'num_chunks': len(chunks),
            'avg_chunk_length': avg_length,
        })
    
    return results

# Evaluate
chunk_sizes = [50, 100, 200, 300, 500, 1000]
results = evaluate_chunk_size(sample_doc * 5, chunk_sizes)  # Repeat for more data

print("📏 Chunk Size Analysis\n")
for r in results:
    print(f"Size: {r['chunk_size']:4} → {r['num_chunks']} chunks (avg {r['avg_chunk_length']:.0f} chars)")

print("\n💡 Sweet spot is usually 200-500 tokens (300-750 chars)")

## ✅ Summary

### Chunking Strategies:

**1. Fixed-size** (Simple but crude)
```python
# Pros: Fast, predictable
# Cons: May split mid-sentence
chunk_size=500, overlap=50
```

**2. Recursive** (Recommended!)
```python
# Pros: Respects structure
# Cons: Slightly slower
RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50,
    separators=["\n\n", "\n", ". ", " ", ""]
)
```

**3. Semantic** (Best quality)
```python
# Pros: Topic-aware
# Cons: Expensive, variable size
semantic_chunker.chunk(text)
```

### Optimal Settings:

| Use Case | Chunk Size | Overlap | Why |
|----------|------------|---------|-----|
| **Q&A** | 300-500 | 50-100 | Complete answers |
| **Summarization** | 500-1000 | 100-200 | More context |
| **Code** | 200-400 | 30-50 | Function-level |
| **Legal** | 400-600 | 100-150 | Preserve context |

### Overlap Rules:

```python
# Overlap = 10-20% of chunk size
chunk_size=500 → overlap=50-100

# Why overlap?
✅ Prevents information loss at boundaries
✅ Helps with context continuity
✅ Improves retrieval recall
```

### Next: `05_rag_systems/09_query_expansion.ipynb`